In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install auto-round --break-system-packages

In [1]:
import os
import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, notebook_login, get_token
from transformers import AutoModelForImageTextToText, AutoProcessor
from safetensors import safe_open
import safetensors.torch as st
import gc
import json

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.11.0+cu128
CUDA Available: True
CUDA Version: 12.8
GPU Name: NVIDIA L40S
VRAM: 44.5 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "google/gemma-4-E4B-it"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound-e4b"
LOCAL_PATH = "./local_model-e4b"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Fetching 9 files:   0%|                                  | 0/9 [00:00<?, ?it/s]Still waiting to acquire lock on /workspace/local_model-e4b/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model-e4b/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model-e4b/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model-e4b/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 9 files: 100%|██████████████████████████| 9/9 [00:27<00:00,  3.02s/it]
Download complete: 100%|███████████████████| 16.0G/16.0G [00:27<00:00, 588MB/s]✓ Downloaded
  path: /workspace/local_model-e4b
Download complete: 100%|███████████████████| 16.0G/16.0G [00:27<00:00, 588MB/s]


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [8]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (o_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=Fals

In [ ]:
# layer_config = {}
# for name, _ in model.named_modules():
#     if name.startswith(("model.vision_tower", "model.audio_tower",
#                         "model.multi_modal_projector", "model.audio_projector")):
#         layer_config[name] = {"bits": 32}

# print(f"Skipping {len(layer_config)} non-LM modules")


In [9]:
TUNING_CONFIG = {
    "group_size": 128,
    "sym": True,
    "iters": 800,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 2,  # Faster on 48GB VRAM
    "seqlen": 2048,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
    # "layer_config": layer_config
}

In [10]:
def push_to_hub(local_dir, repo_name, token):
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")
    try:
        api = HfApi()
        create_repo(full_repo_id, repo_type="model", exist_ok=True, private=False, token=token)
        api.upload_folder(folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token)
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [11]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-06-07 14:00:43 INFO entry.py L587: Using MLLM mode for multimodal model.


In [12]:
# SINGLE CALL to save autoround formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round", inplace=True
)

2026-06-07 14:00:49 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): 
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-06-07 14:00:49 WARNING special_model_handler.py L383: Applying a monkey patch to Gemma4 to reduce RAM usage. This patch has only been validated with limited Transformers versions. Proceed with caution.
2026-06-07 14:00:49 INFO data_driven.py L662: start to cache block inputs
2026-06-07 14:00:49 INFO mllm.py L83: Using MLLM template: gemma4
2026-06-07 14:00:49 INFO calib_dataset.py L977: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-06-07 14:01:23 WARNING logging.py L340: Please note that 'shared_kv_states' key is not currently used in quantization fine-tuning.
2026-06-07 14:01:47 INFO data_driven.py L685: caching done
Quantizing model.language_model.layers.0:   0%|          | 0/42 [00:01<?, ?it/s]/usr/local/lib/python3

(Gemma4ForConditionalGeneration(
   (model): Gemma4Model(
     (vision_tower): Gemma4VisionModel(
       (patch_embedder): Gemma4VisionPatchEmbedder(
         (input_proj): Linear(in_features=768, out_features=768, bias=False)
       )
       (encoder): Gemma4VisionEncoder(
         (rotary_emb): Gemma4VisionRotaryEmbedding()
         (layers): ModuleList(
           (0-15): 16 x Gemma4VisionEncoderLayer(
             (self_attn): Gemma4VisionAttention(
               (q_proj): Gemma4ClippableLinear(
                 (linear): Linear(in_features=768, out_features=768, bias=False)
               )
               (k_proj): Gemma4ClippableLinear(
                 (linear): Linear(in_features=768, out_features=768, bias=False)
               )
               (v_proj): Gemma4ClippableLinear(
                 (linear): Linear(in_features=768, out_features=768, bias=False)
               )
               (o_proj): Gemma4ClippableLinear(
                 (linear): Linear(in_features=768, out_f

In [13]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [15]:
if hf_token:
    # Push auto_round format
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-e4b-w4g128"),
        f"{base_name}-W4A16-AutoRound",
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")



[Hub] Pushing ./AutoRound-e4b/local_model-e4b-w4g128 to Vishva007/gemma-4-E4B-it-W4A16-AutoRound...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/gemma-4-E4B-it-W4A16-AutoRound
